# 🎬 Movie Recommender — Tkinter Desktop UI

A native desktop GUI (instead of the Streamlit web app) for the same
content-based movie recommender. Pick a movie from the searchable dropdown,
click **Recommend**, and 5 similar movies (with posters) appear below.

**Requirements before running this notebook:**
1. Run `movie_recommender_system.ipynb` first (or otherwise produce
   `model/movie_list.pkl` and `model/similarity.pkl`) — this UI just loads
   that pre-computed model, it doesn't retrain anything.
2. `pip install pillow requests` (Pillow is needed to display poster images
   inside a Tkinter window; plain Tkinter can only show `.gif`/`.png` via a
   more limited `PhotoImage`, so we use Pillow for JPEG poster support).
3. Run this notebook **locally** (Jupyter Desktop, VS Code, or `jupyter
   notebook` from a terminal) — Tkinter opens a real OS window, so it won't
   work in a browser-only / headless / remote Jupyter environment (e.g.
   Google Colab, most cloud notebook servers).

Running the last cell opens the window; closing it (or interrupting the
kernel) ends the app.


## 1. Imports

- `tkinter` / `ttk` — the GUI toolkit (`ttk` gives the more modern-looking
  themed widgets: `Combobox`, `Button`, etc.).
- `PIL.Image` / `PIL.ImageTk` — decode a poster JPEG downloaded from TMDB and
  convert it into a format Tkinter can display.
- `requests` — download poster images over HTTP.
- `pickle` — load the pre-trained `movies` dataframe and `similarity` matrix.
- `io.BytesIO` — lets us feed downloaded image bytes straight into Pillow
  without writing a temp file to disk.
- `threading` — poster downloads happen on a background thread so the UI
  doesn't freeze while waiting on the network.


In [1]:
import io
import pickle
import threading
import tkinter as tk
from tkinter import ttk, messagebox

import requests
from PIL import Image, ImageTk


## 2. Config & model loading

Same TMDB demo API key used in the Streamlit app. Swap in your own key from
https://www.themoviedb.org/settings/api if this one stops working, or set the
`TMDB_API_KEY` environment variable and read it with `os.environ`.


In [2]:
TMDB_API_KEY = "8265bd1679663a7ea12ac168da84d2e8"
POSTER_SIZE = (150, 225)  # width, height — TMDB's w500 posters are 2:3

movies = pickle.load(open("model/movie_list.pkl", "rb"))
similarity = pickle.load(open("model/similarity.pkl", "rb"))
movie_titles = sorted(movies["title"].tolist())

print(f"Loaded {len(movie_titles)} movies.")


Loaded 4806 movies.


## 3. Poster fetching

Downloads a movie's poster from TMDB and returns it as a `PIL.Image` sized to
`POSTER_SIZE`. On any failure (missing poster, network error, bad response) it
returns a plain gray placeholder instead of crashing the app.


In [3]:
def fetch_poster_image(movie_id: int) -> Image.Image:
    try:
        url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={TMDB_API_KEY}&language=en-US"
        resp = requests.get(url, timeout=5)
        resp.raise_for_status()
        poster_path = resp.json().get("poster_path")
        if not poster_path:
            raise ValueError("no poster_path")

        img_resp = requests.get(f"https://image.tmdb.org/t/p/w500{poster_path}", timeout=5)
        img_resp.raise_for_status()
        img = Image.open(io.BytesIO(img_resp.content)).convert("RGB")
    except Exception:
        img = Image.new("RGB", POSTER_SIZE, color=(60, 60, 60))

    return img.resize(POSTER_SIZE)


## 4. Recommendation logic

Identical to the Streamlit app / notebook: look up the chosen movie's row,
rank every other movie by cosine similarity, return the top 5
`(title, movie_id)` pairs.


In [4]:
def get_recommendations(movie_title: str):
    index = movies[movies["title"] == movie_title].index[0]
    distances = sorted(enumerate(similarity[index]), reverse=True, key=lambda x: x[1])
    top5 = distances[1:6]
    return [(movies.iloc[i].title, movies.iloc[i].movie_id) for i, _ in top5]


## 5. Build the window

Layout:
- Title label at the top.
- A searchable `ttk.Combobox` (type to filter the movie list) + a
  **Recommend** button.
- A row of 5 poster panels (image + title), populated after clicking
  Recommend.

Posters are fetched on a background `threading.Thread` so clicking
**Recommend** doesn't freeze the window while 5 network requests run; when the
thread finishes it hands the images back to the main thread with
`root.after(...)`, which is the safe way to touch Tkinter widgets from another
thread.


In [5]:
root = tk.Tk()
root.title("Movie Recommender")
root.geometry("900x420")
root.resizable(False, False)

tk.Label(root, text="Movie Recommender System", font=("Segoe UI", 18, "bold")).pack(pady=(15, 10))

search_frame = tk.Frame(root)
search_frame.pack(pady=5)

selected_movie = tk.StringVar()
combo = ttk.Combobox(search_frame, textvariable=selected_movie, values=movie_titles, width=50)
combo.pack(side=tk.LEFT, padx=(0, 10))
if movie_titles:
    combo.set(movie_titles[0])


def filter_movies(event=None):
    typed = combo.get().lower()
    if typed == "":
        combo["values"] = movie_titles
    else:
        combo["values"] = [t for t in movie_titles if typed in t.lower()]


combo.bind("<KeyRelease>", filter_movies)

results_frame = tk.Frame(root)
results_frame.pack(pady=20)

poster_labels = []
title_labels = []
for i in range(5):
    col = tk.Frame(results_frame)
    col.grid(row=0, column=i, padx=10)
    img_label = tk.Label(col)
    img_label.pack()
    txt_label = tk.Label(col, text="", wraplength=150, justify="center", font=("Segoe UI", 9))
    txt_label.pack(pady=(5, 0))
    poster_labels.append(img_label)
    title_labels.append(txt_label)

status_label = tk.Label(root, text="", fg="gray")
status_label.pack()


def render_results(recs, images):
    for label, title_label, (title, _), img in zip(poster_labels, title_labels, recs, images):
        photo = ImageTk.PhotoImage(img)
        label.configure(image=photo)
        label.image = photo  # keep a reference, otherwise Tkinter garbage-collects it
        title_label.configure(text=title)
    status_label.configure(text="")
    recommend_button.configure(state=tk.NORMAL)


def fetch_in_background(recs):
    images = [fetch_poster_image(movie_id) for _, movie_id in recs]
    root.after(0, render_results, recs, images)


def on_recommend():
    movie = selected_movie.get()
    if movie not in movie_titles:
        messagebox.showwarning("Movie not found", "Please pick a movie from the dropdown list.")
        return

    recommend_button.configure(state=tk.DISABLED)
    status_label.configure(text="Fetching recommendations…")
    for label, title_label in zip(poster_labels, title_labels):
        label.configure(image="")
        title_label.configure(text="")

    recs = get_recommendations(movie)
    threading.Thread(target=fetch_in_background, args=(recs,), daemon=True).start()


recommend_button = tk.Button(search_frame, text="Recommend", command=on_recommend)
recommend_button.pack(side=tk.LEFT)


## 6. Run the app

`root.mainloop()` blocks and hands control to Tkinter's event loop — the
window stays open and responsive until you close it. Run this cell last.


In [ ]:
root.mainloop()